In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import XGBRegressor
from xgboost.callback import TrainingCallback
from sklearn.metrics import mean_absolute_error
from xgboost.callback import TrainingCallback
from xgboost import DMatrix

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/car-trader/car-trader-dataset/cars-data-cleaned.csv')
df = df.drop('year', axis=1)
df.head()

,car-maker,mileage-km,fuel-type,transmission-type,price-rwf,car-age
0,toyota,141520,hybrid,automatic,13528556,9
1,bmw,139091,gasoline,automatic,27587930,10
2,honda,258093,gasoline,automatic,7376030,19
3,hyundai,147560,hybrid,automatic,24723848,9
4,ford,284431,gasoline,automatic,2554924,24


In [ ]:
# Features
x = df.drop('price-rwf', axis=1)

# Target
y = df['price-rwf']

# Separate x_temp = 80% # x_test = 20%
x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

#Split that 80% into TRAIN (80% of 80% = 64%) and VAL (20% of 80% = 16%)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.2, random_state=42)

In [ ]:
# one hot encode
x_train = pd.get_dummies(x_train, columns=['car-maker','fuel-type', 'transmission-type'])
x_val = pd.get_dummies(x_val, columns=['car-maker','fuel-type', 'transmission-type']).reindex(columns=x_train.columns, fill_value=0)
x_test = pd.get_dummies(x_test, columns=['car-maker','fuel-type', 'transmission-type']).reindex(columns=x_train.columns, fill_value=0)

In [ ]:
#
class MAECallback(TrainingCallback):
    def after_iteration(self, model, epoch, evals_log):
        dval = DMatrix(x_val)
        y_pred = model.predict(dval)
        mae = mean_absolute_error(y_val, y_pred)
        print(f"[{epoch}] Validation MAE: {mae:,.0f} RWF")
        return False

In [ ]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,  # Try 8-10
    min_child_weight=3,  # Prevent overfitting
    subsample=0.8,  # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    reg_alpha=0.1,  # L1 regularization
    reg_lambda=1.0,  # L2 regularization
    random_state=42,
    early_stopping_rounds=50
)

model.fit(
    x_train, y_train,
    callbacks=[early_stopping, RealMAECallback()],
    verbose=False
)

[0] Validation MAE: 12,982,598 RWF
[1] Validation MAE: 11,898,922 RWF
[2] Validation MAE: 10,958,222 RWF
[3] Validation MAE: 10,146,884 RWF
[4] Validation MAE: 9,450,283 RWF
[5] Validation MAE: 8,854,822 RWF
[6] Validation MAE: 8,347,104 RWF
[7] Validation MAE: 7,914,652 RWF
[8] Validation MAE: 7,545,866 RWF
[9] Validation MAE: 7,231,608 RWF
[10] Validation MAE: 6,963,564 RWF
[11] Validation MAE: 6,734,648 RWF
[12] Validation MAE: 6,539,341 RWF
[13] Validation MAE: 6,372,521 RWF
[14] Validation MAE: 6,230,014 RWF
[15] Validation MAE: 6,108,193 RWF
[16] Validation MAE: 6,003,973 RWF
[17] Validation MAE: 5,915,008 RWF
[18] Validation MAE: 5,839,132 RWF
[19] Validation MAE: 5,774,527 RWF
[20] Validation MAE: 5,719,438 RWF
[21] Validation MAE: 5,672,602 RWF
[22] Validation MAE: 5,632,925 RWF
[23] Validation MAE: 5,599,338 RWF
[24] Validation MAE: 5,570,826 RWF
[25] Validation MAE: 5,546,753 RWF
[26] Validation MAE: 5,526,476 RWF
[27] Validation MAE: 5,509,414 RWF
[28] Validation MAE: 5,495

XGBRegressor(base_score=None, booster=None,
             callbacks=[<__main__.MAECallback object at 0x7c16f57f1850>],
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=50,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)